In [15]:
from datetime import datetime, timedelta
import json
import uuid
import random 
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import cultpass

# Udahub Accounts

## Cultpass Database

**Init DB**

In [16]:
cultpass_db = "data/external/cultpass.db"

In [17]:
reset_db(cultpass_db)

✅ Removed existing data/external/cultpass.db
✅ Database file removed: data/external/cultpass.db


In [18]:
engine = create_engine(f"sqlite:///{cultpass_db}", echo=False)
cultpass.Base.metadata.create_all(engine)

**Experiences**

In [19]:
experience_data = []

with open('data/external/cultpass_experiences.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        experience_data.append(json.loads(line))

In [20]:
experience_data

[{'title': 'Carnival History Tour in Olinda',
  'description': "Discover the origins and vibrant traditions of Pernambuco's Carnival.",
  'location': 'Pernambuco, Brazil'},
 {'title': 'Sunset Paddleboarding',
  'description': 'Glide across calm waters at golden hour with all gear included.',
  'location': 'Santa Catarina, Brazil'},
 {'title': 'Pelourinho Colonial Walk',
  'description': 'Wander through colorful streets and learn about Afro-Brazilian history.',
  'location': 'Bahia, Brazil'},
 {'title': 'Samba Night at Lapa',
  'description': 'Dance the night away at a traditional samba club in the Lapa arches.',
  'location': 'Rio de Janeiro, Brazil'},
 {'title': 'Christ the Redeemer Experience',
  'description': 'Take a guided trip to one of the New Seven Wonders of the World with historical context.',
  'location': 'Rio de Janeiro, Brazil'},
 {'title': 'Modern Art at MASP',
  'description': 'Enjoy a guided visit to the São Paulo Museum of Art with insights into its top collections.',

In [21]:
with get_session(engine) as session:
    experiences = []

    for idx, experience in enumerate(experience_data):
        exp = cultpass.Experience(
            experience_id=str(uuid.uuid4())[:6],
            title=experience["title"],
            description=experience["description"],
            location=experience["location"],
            when=datetime.now() + timedelta(days=idx+1),
            slots_available=random.randint(1,30),
            is_premium=(idx % 2 == 0)
        )
        experiences.append(exp)

    session.add_all(experiences)

**User**

In [22]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [23]:
cultpass_users

[{'id': 'a4ab87',
  'name': 'Alice Kingsley',
  'email': 'alice.kingsley@wonderland.com',
  'is_blocked': True},
 {'id': 'f556c0',
  'name': 'Bob Stone',
  'email': 'bob.stone@granite.com',
  'is_blocked': False},
 {'id': '88382b',
  'name': 'Cathy Bloom',
  'email': 'cathy.bloom@florals.org',
  'is_blocked': False},
 {'id': '888fb2',
  'name': 'David Noir',
  'email': 'david.noir@shadowmail.com',
  'is_blocked': True},
 {'id': 'f1f10d',
  'name': 'Eva Green',
  'email': 'eva.green@ecosoul.net',
  'is_blocked': False},
 {'id': 'e6376d',
  'name': 'Frank Ocean',
  'email': 'frank.ocean@seawaves.io',
  'is_blocked': False}]

In [24]:
with get_session(engine) as session:
    db_users = []
    for user_info in cultpass_users:
        user = cultpass.User(
            user_id=user_info["id"],
            full_name=user_info["name"],
            email=user_info["email"],
            is_blocked=user_info["is_blocked"],
            created_at=datetime.now()
        )
        db_users.append(user)
    session.add_all(db_users) 

**Subscription**

In [25]:
with get_session(engine) as session:
    subscriptions = []
    for user_info in cultpass_users:
        subscription = cultpass.Subscription(
            subscription_id=str(uuid.uuid4())[:6],
            user_id=user_info["id"],
            status=random.choice(["active", "cancelled"]),
            tier=random.choice(["basic", "premium"]),
            monthly_quota=random.randint(2,10),
            started_at=datetime.now()
        )
        subscriptions.append(subscription)

    session.add_all(subscriptions)

**Reservation**

In [26]:
# Applicable to `cultpass_users[0]` at the moment

with get_session(engine) as session:
    experience_ids = [
        exp.experience_id 
        for exp 
        in session.query(cultpass.Experience).all()
    ]

    reservation1 = cultpass.Reservation(
        reservation_id=str(uuid.uuid4())[:6],
        user_id=cultpass_users[0]["id"],
        experience_id=random.choice(experience_ids),
        status="reserved",
    )

    reservation2 = cultpass.Reservation(
        reservation_id=str(uuid.uuid4())[:6],
        user_id=cultpass_users[0]["id"],
        experience_id=random.choice(experience_ids),
        status="reserved",
    )

    session.add_all([reservation1, reservation2])

In [27]:
# Add more reservations for all users with diverse statuses
# This creates a realistic dataset for testing the multi-agent system

with get_session(engine) as session:
    # Get all users and experiences
    all_users = session.query(cultpass.User).all()
    all_experiences = session.query(cultpass.Experience).all()
    experience_ids = [exp.experience_id for exp in all_experiences]
    
    reservations = []
    
    # Create reservations for each user
    for user in all_users:
        # Get user's subscription to determine quota
        subscription = session.query(cultpass.Subscription).filter_by(user_id=user.user_id).first()
        
        if subscription and subscription.status == "active":
            # Active users get 2-5 reservations
            num_reservations = random.randint(2, min(5, subscription.monthly_quota))
        else:
            # Inactive/cancelled users get 0-2 reservations
            num_reservations = random.randint(0, 2)
        
        for i in range(num_reservations):
            # Vary reservation statuses for realistic testing
            if i == 0:
                # First reservation is usually active
                status = "reserved"
            else:
                # Others have mixed statuses
                status = random.choice([
                    "reserved",      # 40%
                    "reserved",
                    "cancelled",     # 30%
                    "cancelled",
                    "completed",     # 30%
                    "completed"
                ])
            
            reservation = cultpass.Reservation(
                reservation_id=str(uuid.uuid4())[:6],
                user_id=user.user_id,
                experience_id=random.choice(experience_ids),
                status=status
            )
            reservations.append(reservation)
    
    # Add all reservations
    session.add_all(reservations)
    
    print(f"✅ Created {len(reservations)} reservations for {len(all_users)} users")

✅ Created 12 reservations for 6 users


# Tests

In [28]:
with get_session(engine) as session:
    users = session.query(cultpass.User).all()
    for user in users:
        print(user)

<User(user_id='a4ab87', email='alice.kingsley@wonderland.com', is_blocked=True)>
<User(user_id='f556c0', email='bob.stone@granite.com', is_blocked=False)>
<User(user_id='88382b', email='cathy.bloom@florals.org', is_blocked=False)>
<User(user_id='888fb2', email='david.noir@shadowmail.com', is_blocked=True)>
<User(user_id='f1f10d', email='eva.green@ecosoul.net', is_blocked=False)>
<User(user_id='e6376d', email='frank.ocean@seawaves.io', is_blocked=False)>


In [29]:
with get_session(engine) as session:
    users = session.query(cultpass.User).all()
    for user in users:
        print(user.subscription)

<Subscription(subscription_id='70440d', user_id='a4ab87', status='active', tier='premium')>
<Subscription(subscription_id='5a29a6', user_id='f556c0', status='active', tier='basic')>
<Subscription(subscription_id='a2da34', user_id='88382b', status='cancelled', tier='premium')>
<Subscription(subscription_id='025e18', user_id='888fb2', status='active', tier='premium')>
<Subscription(subscription_id='ae73cc', user_id='f1f10d', status='cancelled', tier='premium')>
<Subscription(subscription_id='af89f3', user_id='e6376d', status='active', tier='basic')>


In [30]:
with get_session(engine) as session:
    experiences = session.query(cultpass.Experience).all()
    for experience in experiences:
        print(experience)

<Experience(experience_id='1c1f21', title='Carnival History Tour in Olinda', when='2025-11-10 10:18:10.344079')>
<Experience(experience_id='a78d4f', title='Sunset Paddleboarding', when='2025-11-11 10:18:10.344401')>
<Experience(experience_id='31914a', title='Pelourinho Colonial Walk', when='2025-11-12 10:18:10.344539')>
<Experience(experience_id='b12fc4', title='Samba Night at Lapa', when='2025-11-13 10:18:10.344663')>
<Experience(experience_id='599042', title='Christ the Redeemer Experience', when='2025-11-14 10:18:10.344751')>
<Experience(experience_id='64dbe6', title='Modern Art at MASP', when='2025-11-15 10:18:10.344818')>
<Experience(experience_id='695cd8', title='Ibirapuera Park Bike Ride', when='2025-11-16 10:18:10.344872')>
